# 07.04 · Workflow patterns

**Module 07 · LangGraph Fundamentals** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 150–180 min |
| 📋 Prerequisites | Lessons 07.02–07.03. |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Implement** the five canonical workflows: prompt chaining (with a gate), parallelization, routing, orchestrator-worker, evaluator-optimizer.
2. **Build an agent loop by hand** (model node + `ToolNode` + a conditional edge), and see that it's what `create_agent` builds for you.
3. **Choose** a pattern for a task: predictability, cost, latency and flexibility.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents) | All patterns of this lesson, with diagrams; workflows (predetermined paths) vs agents (dynamic) |
| [Use the Graph API](https://docs.langchain.com/oss/python/langgraph/use-graph-api) | `Send`, parallel branches, waiting for branches |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2"

**You should now see** nothing, or *"Note: you may need to restart the kernel to use updated packages."* Restart only if a package was **upgraded** after you had already imported it (Lesson 00.07). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.03).

In [ ]:
from importlib import metadata

for package in ("langgraph", "langchain", "langchain-core",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/07-langgraph-fundamentals/).
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import the course package (installed editable in Lesson 00.02; src/ fallback otherwise).
try:
    import mastery
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import mastery
    print("ℹ️ Loaded mastery from src/. Fix once: python -m pip install -e \".[dev]\"")
print("Repo root:", REPO_ROOT)

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (07.07 reuses Module 06): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

> **Offline models in this lesson.** Each pattern runs with a small scripted model (`ScriptedChatModel` with a function that answers from the prompt), so you see the **control flow** without a key. A 💸 cell at the end runs two patterns with `gpt-oss-20b`. Every builder takes the model as an argument: the graph code doesn't change.

The docs' distinction, in one line: **workflows** have predetermined code paths; **agents** decide their own path and tools. Patterns 1–5 are workflows; Part 6 is an agent.

In [ ]:
# 🧪 Offline: shared helpers: a scripted model that echoes, and simple structured-output stand-ins.
import operator
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import AIMessage
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from pydantic import BaseModel, Field

from mastery.graphs import show_graph
from mastery.testing import ScriptedChatModel


def scripted(fn):
    """A chat model whose reply is fn(prompt_text) (a str, or a pydantic object for structured output)."""
    def respond(messages):
        out = fn(messages[-1].text)
        if isinstance(out, BaseModel):
            return AIMessage("", tool_calls=[{"name": type(out).__name__, "id": "s1", "args": out.model_dump()}])
        return AIMessage(out)
    return ScriptedChatModel(responses=[respond], repeat_last=True)


print("helpers ready")

---
## Pattern 1 · Prompt chaining, with a gate

Each LLM call works on the previous one's output. A **gate** (plain code) checks an intermediate result and stops early or takes another path. Use it when a task splits cleanly into fixed steps: draft → check → improve.

```mermaid
flowchart LR
    START([START]) --> draft_tagline --> gate{has a number?}
    gate -->|no| add_fact --> polish --> END([END])
    gate -->|yes| polish
```

In [ ]:
# 🧪 Offline: draft → gate (code) → optionally add a fact → polish.
class Tagline(TypedDict, total=False):
    product: str
    text: str


def build_chain(model):
    def draft_tagline(state: Tagline) -> dict:
        return {"text": model.invoke(f"Write a tagline for {state['product']}.").text}

    def add_fact(state: Tagline) -> dict:
        return {"text": model.invoke(f"Add one concrete number to: {state['text']}").text}

    def polish(state: Tagline) -> dict:
        return {"text": model.invoke(f"Polish, keep it under 12 words: {state['text']}").text}

    def gate(state: Tagline) -> str:
        return "polish" if any(ch.isdigit() for ch in state["text"]) else "add_fact"

    b = StateGraph(Tagline)
    for n in (draft_tagline, add_fact, polish):
        b.add_node(n)
    b.add_edge(START, "draft_tagline")
    b.add_conditional_edges("draft_tagline", gate, ["add_fact", "polish"])
    b.add_edge("add_fact", "polish")
    b.add_edge("polish", END)
    return b.compile()


def tagline_writer(prompt: str) -> str:
    if prompt.startswith("Write"):
        return "The Nimbus Mini: weather, simply."
    if prompt.startswith("Add"):
        return prompt.split(": ", 1)[1].replace("simply", "every 60 seconds")
    return prompt.split(": ", 1)[1]


chain_graph = build_chain(scripted(tagline_writer))
print(chain_graph.invoke({"product": "Nimbus Mini"})["text"])

**You should now see** `The Nimbus Mini: weather, every 60 seconds.` The draft had no number, so the gate sent it through `add_fact`.

---
## Pattern 2 · Parallelization

Independent subtasks run **at the same time**, then an aggregator combines them. Two flavours: **sectioning** (different subtasks) and **voting** (the same task several times). Here: three checks of a product description, in parallel, then one verdict.

In [ ]:
# 🧪 Offline: three independent checks fan out from START; `verdict` waits for all three.
class Review(TypedDict, total=False):
    text: str
    checks: Annotated[list[str], operator.add]
    verdict: str


def build_parallel(model):
    def make_check(name, question):
        def check(state: Review) -> dict:
            answer = model.invoke(f"{question}\nText: {state['text']}").text
            return {"checks": [f"{name}: {answer}"]}
        check.__name__ = name
        return check

    checks = [make_check("tone", "Is the tone friendly? yes/no"),
              make_check("claims", "Does it make unverifiable claims? yes/no"),
              make_check("length", "Is it under 40 words? yes/no")]

    def verdict(state: Review) -> dict:
        bad = [c for c in state["checks"] if c == "claims: yes" or (not c.startswith("claims") and c.endswith(": no"))]
        return {"verdict": "fix: " + "; ".join(sorted(bad)) if bad else "ok"}

    b = StateGraph(Review)
    for c in checks:
        b.add_node(c)
        b.add_edge(START, c.__name__)
    b.add_node(verdict)
    b.add_edge([c.__name__ for c in checks], "verdict")      # list form: wait for ALL three
    b.add_edge("verdict", END)
    return b.compile()


def checker(prompt: str) -> str:
    text = prompt.split("Text: ", 1)[1]
    if prompt.startswith("Is the tone"):
        return "yes"
    if prompt.startswith("Does it make"):
        return "yes" if "best in the world" in text else "no"
    return "yes" if len(text.split()) < 40 else "no"


parallel_graph = build_parallel(scripted(checker))
print(parallel_graph.invoke({"text": "The best in the world weather station for your garden."})["verdict"])
print(parallel_graph.invoke({"text": "A friendly station that reports rain every minute."})["verdict"])

**You should now see** `fix: claims: yes` and `ok`. The three checks ran in one super-step; `operator.add` collected their results; `add_edge([...], "verdict")` made `verdict` wait for all of them.

---
## Pattern 3 · Routing

Classify the input, then send it to a **specialised** path (different prompt, model or tools). Structured output makes the route a typed value instead of free text.

In [ ]:
# 🧪 Offline: a router with structured output, three specialised handlers.
class Route(BaseModel):
    step: Literal["billing", "technical", "sales"] = Field(description="Which team handles this message")


class Inbox(TypedDict, total=False):
    message: str
    route: str
    answer: str


def build_router(model):
    router = model.with_structured_output(Route)

    def classify(state: Inbox) -> dict:
        return {"route": router.invoke(f"Route this message: {state['message']}").step}

    def billing(state: Inbox) -> dict:
        return {"answer": "Billing: we'll check your invoice."}

    def technical(state: Inbox) -> dict:
        return {"answer": "Support: try restarting the bridge first."}

    def sales(state: Inbox) -> dict:
        return {"answer": "Sales: here are our plans."}

    b = StateGraph(Inbox)
    for n in (classify, billing, technical, sales):
        b.add_node(n)
    b.add_edge(START, "classify")
    b.add_conditional_edges("classify", lambda s: s["route"], ["billing", "technical", "sales"])
    for n in ("billing", "technical", "sales"):
        b.add_edge(n, END)
    return b.compile()


def keyword_route(prompt: str) -> Route:
    text = prompt.lower()
    return Route(step="billing" if "invoice" in text or "charge" in text else "technical" if "offline" in text or "error" in text else "sales")


router_graph = build_router(scripted(keyword_route))
for msg in ("My station shows error E40", "Why was I charged twice?", "Do you sell to schools?"):
    out = router_graph.invoke({"message": msg})
    print(f"{out['route']:<10} {out['answer']}")

**You should now see** `technical`, `billing`, `sales` with the matching answers.

---
## Pattern 4 · Orchestrator-worker

Like parallelization, but the subtasks are **not known in advance**: an orchestrator (an LLM) plans them, workers run one per subtask via `Send`, a synthesizer combines. Project 07's research step is exactly this.

In [ ]:
# 🧪 Offline: plan sections (structured), one worker per section with Send, then synthesize.
class Section(BaseModel):
    name: str
    description: str


class Plan(BaseModel):
    sections: list[Section]


class Report(TypedDict, total=False):
    topic: str
    plan: Plan
    written: Annotated[list[str], operator.add]
    report: str


def build_orchestrator(planner, worker):
    plan_model = planner.with_structured_output(Plan)

    def orchestrator(state: Report) -> dict:
        return {"plan": plan_model.invoke(f"Plan a short report on: {state['topic']}")}

    def assign_workers(state: Report) -> list[Send]:
        return [Send("write_section", {"section": s}) for s in state["plan"].sections]

    def write_section(task: dict) -> dict:
        s = task["section"]
        return {"written": [f"## {s.name}\n" + worker.invoke(f"Write 1 sentence: {s.description}").text]}

    def synthesizer(state: Report) -> dict:
        order = [s.name for s in state["plan"].sections]           # completion order ≠ plan order
        body = sorted(state["written"], key=lambda w: order.index(w.splitlines()[0][3:]))
        return {"report": "\n\n".join(body)}

    b = StateGraph(Report)
    for n in (orchestrator, write_section, synthesizer):
        b.add_node(n)
    b.add_edge(START, "orchestrator")
    b.add_conditional_edges("orchestrator", assign_workers, ["write_section"])
    b.add_edge("write_section", "synthesizer")
    b.add_edge("synthesizer", END)
    return b.compile()


plan_for = lambda prompt: Plan(sections=[Section(name=n, description=f"{n} of {prompt.split(': ', 1)[1]}")
                                         for n in ("Context", "Options", "Recommendation")])
orchestrator_graph = build_orchestrator(scripted(plan_for), scripted(lambda p: p.split(": ", 1)[1].capitalize() + "."))
print(orchestrator_graph.invoke({"topic": "weather stations for schools"})["report"])

> 🌐 Draws the graph as a PNG through mermaid.ink (a public web service; no key, no tokens). Offline or blocked? `show_graph` prints a ⚠️ and shows the Mermaid text instead.

In [ ]:
# 🌐 The orchestrator-worker graph: the dotted edge to write_section is the Send fan-out.
_ = show_graph(orchestrator_graph)

**You should now see** three sections in plan order (Context, Options, Recommendation), then the diagram: `orchestrator -.-> write_section --> synthesizer`.

---
## Pattern 5 · Evaluator-optimizer

One LLM generates, another **evaluates** and gives feedback, loop until it passes **or** a counter runs out. Use it when you have clear evaluation criteria and iteration measurably helps (translation, writing to a style guide, code that must pass checks). `loop_router` from `mastery.graphs` is the exit rule.

In [ ]:
# 🧪 Offline: generate → evaluate → (revise ⟲ | done). Stops on a pass OR after 3 revisions.
from mastery.graphs import loop_router


class Feedback(BaseModel):
    score: int = Field(ge=1, le=10)
    advice: str


class Joke(TypedDict, total=False):
    topic: str
    joke: str
    feedback: Feedback
    iteration: int


def build_evaluator_optimizer(generator, evaluator, max_iterations=3):
    grade = evaluator.with_structured_output(Feedback)

    def generate(state: Joke) -> dict:
        prompt = f"Write a joke about {state['topic']}."
        if state.get("feedback"):
            prompt += f" Previous: {state['joke']} Feedback: {state['feedback'].advice}"
        return {"joke": generator.invoke(prompt).text, "iteration": state.get("iteration", -1) + 1}

    def evaluate(state: Joke) -> dict:
        return {"feedback": grade.invoke(f"Grade this joke 1-10: {state['joke']}")}

    b = StateGraph(Joke)
    b.add_node(generate)
    b.add_node(evaluate)
    b.add_edge(START, "generate")
    b.add_edge("generate", "evaluate")
    b.add_conditional_edges("evaluate", loop_router(score_key="feedback", threshold=7, max_iterations=max_iterations,
                                                    again="generate", done=END), ["generate", END])
    return b.compile()


JOKES = ["Rain is wet.",
         "Why did the rain gauge quit? It was under too much pressure.",
         "My barometer and I broke up: too much pressure, never any high points."]


def joke_writer(prompt: str) -> str:
    previous = prompt.split("Previous: ", 1)[1].split(" Feedback:")[0] if "Previous: " in prompt else None
    return JOKES[min(JOKES.index(previous) + 1, 2)] if previous in JOKES else JOKES[0]


def joke_grader(prompt: str) -> Feedback:
    joke = prompt.split(": ", 1)[1]
    score = 3 if len(joke) < 20 else 6 if "gauge" in joke else 8
    return Feedback(score=score, advice="needs a punchline" if score < 5 else "sharper wordplay")


eo = build_evaluator_optimizer(scripted(joke_writer), scripted(joke_grader))
for mode in eo.stream({"topic": "weather"}, stream_mode="updates"):
    for node, update in mode.items():
        print(f"  {node:<9}", update.get("joke") or f"score={update['feedback'].score} ({update['feedback'].advice})")

**You should now see** three generate/evaluate rounds: score 3 (`needs a punchline`), 6, then 8, and the loop stops because 8 ≥ 7. With a grader that never approves, `max_iterations=3` would stop it after the third revision (try it in Exercise 3).

### What just happened?

The loop has **two** exits and one of them doesn't depend on the model: the counter. That's what makes it safe to put a model in a cycle. Project 07 uses the same structure with a `Critique` schema.

---
## Part 6 · An agent loop, by hand

An agent is a loop where the **model** decides: call tools, or answer. In LangGraph: a model node, a `ToolNode` that runs the requested tools, and a conditional edge that loops while there are tool calls. That is what `create_agent` builds for you (plus middleware, structured output, and more).

In [ ]:
# 🧪 Offline: model node + ToolNode + "any tool calls?" edge. A scripted model asks for one tool, then answers.
from langchain_core.tools import tool
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode


@tool
def price_of(product: str) -> str:
    """Price in USD of an Orbita product (e.g. 'Nimbus Pro')."""
    return {"nimbus pro": "349", "nimbus mini": "89"}.get(product.lower(), "unknown")


@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


TOOLS = [price_of, multiply]


def build_agent(model, max_model_calls=6):
    bound = model.bind_tools(TOOLS)

    def call_model(state: MessagesState) -> dict:
        return {"messages": [bound.invoke(state["messages"])]}

    def should_continue(state: MessagesState) -> str:
        calls = sum(1 for m in state["messages"] if m.type == "ai")
        if state["messages"][-1].tool_calls and calls < max_model_calls:
            return "tools"
        return END

    b = StateGraph(MessagesState)
    b.add_node("model", call_model)
    b.add_node("tools", ToolNode(TOOLS))
    b.add_edge(START, "model")
    b.add_conditional_edges("model", should_continue, ["tools", END])
    b.add_edge("tools", "model")
    return b.compile()


script = [
    AIMessage("", tool_calls=[{"name": "price_of", "args": {"product": "Nimbus Pro"}, "id": "c1"}]),
    AIMessage("", tool_calls=[{"name": "multiply", "args": {"a": 349, "b": 3}, "id": "c2"}]),
    AIMessage("Three Nimbus Pro stations cost US$1047."),
]
hand_agent = build_agent(ScriptedChatModel(responses=script))
out = hand_agent.invoke({"messages": [("user", "How much are 3 Nimbus Pro stations?")]})
for m in out["messages"]:
    print(f"{m.type:<6}", m.content or m.tool_calls[0]["name"] + str(m.tool_calls[0]["args"]))

**You should now see** the transcript: `human`, `ai price_of{'product': 'Nimbus Pro'}`, `tool 349`, `ai multiply{...}`, `tool 1047.0`, `ai Three Nimbus Pro stations cost US$1047.` The `max_model_calls` check in `should_continue` is our hand-made `ModelCallLimitMiddleware`.

| Pattern | Who decides the path | Calls per input | Pick it when |
|---|---|---|---|
| Prompt chaining | you (fixed steps + gates) | fixed | The task splits into known steps |
| Parallelization | you | fixed, concurrent | Independent subtasks; or voting for confidence |
| Routing | a classifier, then you | 1 + handler | Distinct input types need distinct handling |
| Orchestrator-worker | the planner (how many), you (how) | 1 + N + 1 | Subtasks known only at run time |
| Evaluator-optimizer | the evaluator, capped by you | 2 × rounds | Clear criteria, iteration helps |
| Agent | the model | unknown (cap it!) | Open-ended tasks, tool choice can't be scripted |

> 💸 Calls `gpt-oss-20b` a few times (one per model node), a few thousand tokens of your free-tier quota. Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 💸 Live: the router and the evaluator-optimizer with gpt-oss-20b. Same builders, real model.
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    llm = get_model("fast", temperature=0, reasoning_effort="low")
    live_router = build_router(llm)
    for msg in ("My station shows error E40", "Why was I charged twice?", "Do you sell to schools?"):
        print("route:", live_router.invoke({"message": msg})["route"], "|", msg)
    live_eo = build_evaluator_optimizer(get_model("fast", temperature=0.7, reasoning_effort="low"), llm)
    final = live_eo.invoke({"topic": "weather forecasts"})
    print(f"\nafter {final['iteration']} revision(s), score {final['feedback'].score}: {final['joke']}")
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) `technical`, `billing`, `sales`, then a joke with its score after 0–3 revisions. A model grader is often generous (7+ on the first try): Project 07 makes the critic's criteria explicit to get useful feedback.

## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Aggregator runs before all branches finish | Separate `add_edge` per branch with different branch lengths | `add_edge(["a", "b", "c"], "join")`, or `defer=True` on the join node |
| Orchestrator sections come back shuffled | Workers finish in any order | Sort by the plan (as `synthesizer` does) |
| Evaluator-optimizer never stops improving | The grader is never satisfied | `max_iterations` in `loop_router`; plus a `recursion_limit` |
| Hand-made agent loops forever | Model keeps calling tools | A call counter in the router (as above), or `create_agent` + `ModelCallLimitMiddleware` |
| `ToolNode` raises on unknown tool | The model invented a tool name | `ToolNode` returns an error `ToolMessage` by default (`handle_tool_errors`); keep the tool list small and well described |

## ⚠️ Production notes

> ⚠️ **Production note: prefer the simplest pattern that works.**
> Each step up (chain → router → orchestrator → agent) adds calls, latency and variance. Start with a chain, measure, and add flexibility only where the measurements say you need it.

> ⚠️ **Production note: give every LLM step its own budget.**
> Parallel and looping patterns multiply calls. Log model calls and tokens per node (07.06 and Project 07 show how) and alert when a run exceeds its budget.

## 🏋️ Exercises

1. **(Easy)** Add a `"partnership"` route to `build_router` (prompt, node, edge). Test it offline.
2. **(Medium)** Turn the parallel review into **voting**: run the `claims` check 3 times and flag the text if 2 of 3 say yes.
3. **(Medium)** Run `build_evaluator_optimizer` with a grader that always returns score 2. How many times does `generate` run?
4. **(Stretch, 💸)** Rebuild the hand-made agent with `create_agent(llm, TOOLS)` and compare `get_graph().draw_mermaid()` of both.

<details>
<summary><b>Show solutions</b></summary>

**1.** Add `"partnership"` to the `Literal`, a `partnership` node, the edge to `END`, and the name in the `add_conditional_edges` list.

**2.** Three nodes `claims_1..3` from `START` (use `make_check` with different names), then `verdict` counts lines starting with `claims_` that say `yes`.

**3.** Four times: the first draft plus three revisions (`iteration` goes 0 → 3), then `loop_router` sends it to `END` because `iteration >= max_iterations`.

**4.** Both have a `model` node and a `tools` node in a loop; `create_agent`'s graph also shows middleware hooks if you add middleware. Same idea, more built in.
</details>

## ✅ Checkpoint

- [ ] I can implement each of the five workflow patterns and say when to use it.
- [ ] I built an agent loop with `ToolNode` and a stop rule.
- [ ] I used a Mermaid sketch before code and compared it with the drawing.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

The [Workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents) page shows each pattern in **both** APIs (Graph and Functional). Pick one pattern and compare the two versions: which is shorter, which is easier to draw and debug? Keep the note for 07.05's rule of thumb in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Workflows (fixed paths): chaining, parallelization, routing, orchestrator-worker, evaluator-optimizer. Agents: the model chooses the path.
- `Send` powers orchestrator-worker; list-form edges wait for parallel branches; `loop_router` bounds evaluator loops.
- An agent is a model node + `ToolNode` + a loop edge with a stop rule; `create_agent` packages it.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 07.05 · Functional API** → [`07-05-functional-api.ipynb`](07-05-functional-api.ipynb).

[Module 07 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)